In [2]:
# BLOCK 1: Install required libraries
!pip install dash plotly pandas pyngrok dash-bootstrap-components --quiet

In [3]:
# BLOCK 2: Imports
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output, callback
import dash_bootstrap_components as dbc
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print(" All libraries imported successfully")

 All libraries imported successfully


In [4]:
# BLOCK 3: Load and preprocess data
df = pd.read_csv('/content/war_news_events_dataset_updated.csv')

# Parse dates
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.to_period('M').astype(str)
df['Week']  = df['Date'].dt.to_period('W').astype(str)
df['Year']  = df['Date'].dt.year

# Clean column names (strip spaces)
df.columns = df.columns.str.strip()

print(f" Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"   Date range : {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"   Conflicts  : {df['Conflict'].nunique()}")
print(f"   Event types: {df['Event_Type'].nunique()}")
df.head(3)

 Dataset loaded: 300 rows × 12 columns
   Date range : 2023-01-01 → 2024-05-31
   Conflicts  : 5
   Event types: 6


,Date,Country,Conflict,Event_Type,Fatalities,Source,Summary,Latitude,Longitude,Month,Week,Year
0,2023-01-01,Myanmar,Myanmar Conflict,Missile Strike,33,Al Jazeera,Missile Strike reported in Myanmar related to ...,21.1342,96.2032,2023-01,2022-12-26/2023-01-01,2023
1,2023-01-01,Myanmar,Myanmar Conflict,Drone Attack,25,The Guardian,Drone Attack reported in Myanmar related to My...,21.3402,95.2146,2023-01,2022-12-26/2023-01-01,2023
2,2023-01-18,Myanmar,Myanmar Conflict,Ground Combat,28,Associated Press,Ground Combat reported in Myanmar related to M...,21.1504,95.0739,2023-01,2023-01-16/2023-01-22,2023


In [5]:
# BLOCK 4: Consistent color palette for all charts
CONFLICT_COLORS = {
    'Russia–Ukraine War'   : '#e74c3c',
    'Yemen Civil War'      : '#e67e22',
    'Sudan Civil War'      : '#f39c12',
    'Myanmar Conflict'     : '#9b59b6',
    'Israel–Hamas Conflict': '#2980b9',
}

EVENT_COLORS = {
    'Airstrike'             : '#e74c3c',
    'Ground Combat'         : '#e67e22',
    'Missile Strike'        : '#f1c40f',
    'Drone Attack'          : '#2ecc71',
    'Cyber Operation'       : '#3498db',
    'Political Negotiation' : '#9b59b6',
}

BG_COLOR   = '#1a1a2e'   # dark navy background
CARD_COLOR = '#16213e'   # slightly lighter card bg
TEXT_COLOR = '#ecf0f1'

print(" Color palette defined")

 Color palette defined


In [6]:
# BLOCK 5: Chart builder functions (called by callbacks)

DARK_LAYOUT = dict(
    paper_bgcolor=CARD_COLOR,
    plot_bgcolor =CARD_COLOR,
    font_color   =TEXT_COLOR,
    margin=dict(l=40, r=20, t=50, b=40),
)

# ── Chart 1: Global Conflict Map ──────────────────────────────────────────────
# REPLACE entire make_map with this clean version (no numpy jitter bug)
def make_map(filtered_df):
    # Add small manual jitter so overlapping country centroids separate visually
    plot_df = filtered_df.copy()
    import numpy as np
    rng = np.random.default_rng(seed=42)
    plot_df['Latitude']  = plot_df['Latitude']  + rng.uniform(-1.2, 1.2, size=len(plot_df))
    plot_df['Longitude'] = plot_df['Longitude'] + rng.uniform(-1.2, 1.2, size=len(plot_df))

    fig = px.scatter_mapbox(
        plot_df,
        lat='Latitude', lon='Longitude',
        color='Conflict',
        size='Fatalities',
        size_max=40,
        opacity=0.65,
        hover_data={'Country':True, 'Event_Type':True, 'Fatalities':True,
                    'Source':True, 'Date':True, 'Latitude':False, 'Longitude':False},
        color_discrete_map=CONFLICT_COLORS,
        mapbox_style='carto-darkmatter',
        zoom=1.5,
        title='🌍 Global Conflict Map — Event Hotspots',
    )
    fig.update_layout(
        paper_bgcolor=CARD_COLOR, font_color=TEXT_COLOR,
        margin=dict(l=0, r=0, t=50, b=0), height=480,
        legend=dict(bgcolor=CARD_COLOR),
    )
    return fig

print(" make_map fixed")
# ── Chart 2: Conflict Timeline ────────────────────────────────────────────────
def make_timeline(filtered_df, freq='Month'):
    col = 'Month' if freq == 'Month' else 'Week'
    agg = filtered_df.groupby([col, 'Conflict']).size().reset_index(name='Events')
    fig = px.line(
        agg, x=col, y='Events', color='Conflict',
        color_discrete_map=CONFLICT_COLORS,
        markers=True,
        title='📈 Conflict Timeline — Events Over Time',
    )
    fig.update_layout(**DARK_LAYOUT, height=360,
                      xaxis_title=freq, yaxis_title='Number of Events')
    return fig

# ── Chart 3 FIX: Conflict Comparison — separate subplots per metric ───────────
from plotly.subplots import make_subplots

def make_comparison(filtered_df):
    agg = filtered_df.groupby('Conflict').agg(
        Total_Events     =('Event_Type', 'count'),
        Total_Fatalities =('Fatalities', 'sum'),
        Avg_Fatalities   =('Fatalities', 'mean'),
    ).reset_index()

    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=('Total Events', 'Total Fatalities', 'Avg Fatalities / Event'),
        shared_yaxes=False,   # ← INDEPENDENT y-axes — key fix
    )

    palette = ['#e74c3c', '#9b59b6', '#3498db', '#f39c12', '#2ecc71']
    colors = [palette[i % len(palette)] for i, c in enumerate(agg['Conflict'])]

    fig.add_trace(go.Bar(
        x=agg['Conflict'], y=agg['Total_Events'],
        marker_color=colors, showlegend=False,
        text=agg['Total_Events'], textposition='outside',
    ), row=1, col=1)

    fig.add_trace(go.Bar(
        x=agg['Conflict'], y=agg['Total_Fatalities'],
        marker_color=colors, showlegend=False,
        text=agg['Total_Fatalities'], textposition='outside',
    ), row=1, col=2)

    fig.add_trace(go.Bar(
        x=agg['Conflict'], y=agg['Avg_Fatalities'].round(1),
        marker_color=colors, showlegend=False,
        text=agg['Avg_Fatalities'].round(1), textposition='outside',
    ), row=1, col=3)

    fig.update_layout(
        **DARK_LAYOUT,
        height=420,
        title='📊 Conflict Comparison — Independent Metrics',
    )
    # Rotate x labels on all subplots
    fig.update_xaxes(tickangle=-30)

    return fig

# ── Chart 4: Event Type Distribution ─────────────────────────────────────────
def make_event_dist(filtered_df):
    agg = filtered_df.groupby(['Event_Type','Conflict']).size().reset_index(name='Count')
    fig = px.bar(
        agg, x='Event_Type', y='Count', color='Conflict',
        color_discrete_map=CONFLICT_COLORS,
        barmode='stack',
        title='🔫 Event Type Distribution by Conflict',
    )
    fig.update_layout(**DARK_LAYOUT, height=360,
                      xaxis_title='Event Type', yaxis_title='Count',
                      xaxis_tickangle=-15)
    return fig

# ── Chart 5 FIX: Fatality box plot — log scale + all conflicts forced in ──────
def make_fatality(filtered_df):
    # Add a small offset so log scale works (log(0) is undefined)
    plot_df = filtered_df.copy()
    plot_df['Fatalities_log'] = plot_df['Fatalities'].clip(lower=1)

    fig = px.box(
        plot_df,
        x='Conflict', y='Fatalities_log',
        color='Conflict',
        color_discrete_map=CONFLICT_COLORS,
        points='all',           # show every data point, not just outliers
        notched=False,
        title=' Fatality Distribution by Conflict (log scale, all points shown)',
        log_y=True,             # ← log scale so small values are visible
        labels={'Fatalities_log': 'Fatalities (log scale)'},
    )
    fig.update_traces(jitter=0.4, marker_size=5, opacity=0.7)
    fig.update_layout(
        **DARK_LAYOUT,
        height=400,
        showlegend=False,
        xaxis_title='Conflict',
        yaxis_title='Fatalities (log scale)',
        xaxis_tickangle=-20,
    )
    return fig

print(" Fixed chart functions defined — now re-run Block 8 and Block 9")

# ── Chart 6: News Source Reporting Heatmap ───────────────────────────────────
def make_source_heatmap(filtered_df):
    pivot = filtered_df.groupby(['Source','Conflict']).size().reset_index(name='Count')
    mat   = pivot.pivot(index='Source', columns='Conflict', values='Count').fillna(0)
    fig = px.imshow(
        mat,
        color_continuous_scale='YlOrRd',
        title='📰 News Source vs Conflict — Reporting Heatmap',
        aspect='auto',
        text_auto=True,
    )
    fig.update_layout(**DARK_LAYOUT, height=360,
                      coloraxis_colorbar=dict(title='Events'))
    return fig

print(" All chart functions defined")

 make_map fixed
 Fixed chart functions defined — now re-run Block 8 and Block 9
 All chart functions defined


In [7]:
# BLOCK 6: KPI card builder

def make_kpi_cards(filtered_df):
    total_events     = len(filtered_df)
    total_fatalities = filtered_df['Fatalities'].sum()
    avg_fatalities   = filtered_df['Fatalities'].mean()
    countries        = filtered_df['Country'].nunique()

    card_style = {
        'backgroundColor': '#0f3460',
        'borderRadius'   : '10px',
        'padding'        : '15px',
        'textAlign'      : 'center',
        'color'          : TEXT_COLOR,
    }

    def card(label, value, color='#ecf0f1'):
        return dbc.Col(
            html.Div([
                html.H2(f"{value}", style={'color': color, 'margin': '0', 'fontSize':'28px'}),
                html.P(label, style={'margin': '4px 0 0 0', 'fontSize':'12px', 'opacity':'0.75'}),
            ], style=card_style),
            width=3
        )

    return dbc.Row([
        card('Total Events',          f"{total_events:,}",       '#3498db'),
        card('Total Fatalities',      f"{total_fatalities:,}",   '#e74c3c'),
        card('Avg Fatalities/Event',  f"{avg_fatalities:.1f}",   '#f39c12'),
        card('Countries Affected',    f"{countries}",             '#2ecc71'),
    ], className='mb-3')

print(" KPI card builder defined")

 KPI card builder defined


In [8]:
# BLOCK 7: Dash app layout

app = Dash(
    __name__,
    external_stylesheets=[dbc.themes.DARKLY],
    suppress_callback_exceptions=True,
)

# Dropdown options
conflict_options = [{'label': c, 'value': c} for c in sorted(df['Conflict'].unique())]
event_options    = [{'label': e, 'value': e} for e in sorted(df['Event_Type'].unique())]
freq_options     = [{'label': 'Monthly', 'value': 'Month'},
                    {'label': 'Weekly',  'value': 'Week'}]

# Date min/max
date_min = df['Date'].min().date()
date_max = df['Date'].max().date()

HEADER = dbc.Navbar(
    dbc.Container([
        html.H3(" War News Analytical Dashboard",
                style={'color':'white','margin':0,'fontWeight':'bold'}),
        html.Small("B23CM1044 | Data Visualization Assignment 04",
                   style={'color':'#aaa'}),
    ]),
    color='#0f3460', dark=True, className='mb-3',
)

FILTERS = dbc.Card(
    dbc.CardBody([
        html.H6(" Dashboard Controls", style={'color': TEXT_COLOR}),
        dbc.Row([
            dbc.Col([
                html.Label("Conflict", style={'color': TEXT_COLOR, 'fontSize':'12px'}),
                dcc.Dropdown(
                    id='conflict-filter',
                    options=conflict_options,
                    multi=True,
                    placeholder='All Conflicts',
                    style={'backgroundColor':'#1a1a2e','color':'white'},
                ),
            ], md=4),
            dbc.Col([
                html.Label("Event Type", style={'color': TEXT_COLOR, 'fontSize':'12px'}),
                dcc.Dropdown(
                    id='event-filter',
                    options=event_options,
                    multi=True,
                    placeholder='All Event Types',
                    style={'backgroundColor':'#1a1a2e','color':'white'},
                ),
            ], md=4),
            dbc.Col([
                html.Label("Date Range", style={'color': TEXT_COLOR, 'fontSize':'12px'}),
                dcc.DatePickerRange(
                    id='date-filter',
                    min_date_allowed=date_min,
                    max_date_allowed=date_max,
                    start_date=date_min,
                    end_date=date_max,
                    display_format='YYYY-MM-DD',
                    style={'fontSize':'12px'},
                ),
            ], md=4),
        ]),
        dbc.Row([
            dbc.Col([
                html.Label("Timeline Frequency", style={'color': TEXT_COLOR, 'fontSize':'12px'}),
                dcc.RadioItems(
                    id='freq-filter',
                    options=freq_options,
                    value='Month',
                    inline=True,
                    style={'color': TEXT_COLOR, 'fontSize':'13px'},
                ),
            ], md=4),
        ], className='mt-2'),
    ]),
    style={'backgroundColor': CARD_COLOR, 'border':'1px solid #2c3e50'},
    className='mb-3',
)

def section_card(graph_id, title=''):
    return dbc.Card(
        dbc.CardBody([dcc.Graph(id=graph_id, config={'displayModeBar': False})]),
        style={'backgroundColor': CARD_COLOR, 'border':'1px solid #2c3e50'},
        className='mb-3',
    )

app.layout = dbc.Container([
    HEADER,
    FILTERS,

    # KPI row
    html.Div(id='kpi-row', className='mb-3'),

    # Row 1: Map (full width)
    dbc.Row([
        dbc.Col(section_card('map-chart'), width=12),
    ]),

    # Row 2: Timeline + Comparison
    dbc.Row([
        dbc.Col(section_card('timeline-chart'), md=6),
        dbc.Col(section_card('comparison-chart'), md=6),
    ]),

    # Row 3: Event dist + Fatality
    dbc.Row([
        dbc.Col(section_card('event-dist-chart'), md=6),
        dbc.Col(section_card('fatality-chart'), md=6),
    ]),

    # Row 4: Source heatmap (full width)
    dbc.Row([
        dbc.Col(section_card('source-heatmap'), width=12),
    ]),

    # Footer
    html.Hr(style={'borderColor':'#2c3e50'}),
    html.P("B23CM1044 | DV Assignment 04 | War News Analytical Dashboard",
           style={'color':'#666','textAlign':'center','fontSize':'11px'}),

], fluid=True, style={'backgroundColor': BG_COLOR, 'minHeight':'100vh', 'padding':'10px'})

print(" App layout defined")

 App layout defined


In [9]:
# BLOCK 8: Callbacks — all charts update together from filters

@app.callback(
    Output('kpi-row',          'children'),
    Output('map-chart',        'figure'),
    Output('timeline-chart',   'figure'),
    Output('comparison-chart', 'figure'),
    Output('event-dist-chart', 'figure'),
    Output('fatality-chart',   'figure'),
    Output('source-heatmap',   'figure'),
    Input('conflict-filter',   'value'),
    Input('event-filter',      'value'),
    Input('date-filter',       'start_date'),
    Input('date-filter',       'end_date'),
    Input('freq-filter',       'value'),
)
def update_dashboard(conflicts, events, start_date, end_date, freq):
    filtered = df.copy()

    # Apply filters
    if conflicts:
        filtered = filtered[filtered['Conflict'].isin(conflicts)]
    if events:
        filtered = filtered[filtered['Event_Type'].isin(events)]
    if start_date:
        filtered = filtered[filtered['Date'] >= pd.to_datetime(start_date)]
    if end_date:
        filtered = filtered[filtered['Date'] <= pd.to_datetime(end_date)]

    # Handle empty filter result gracefully
    if filtered.empty:
        empty_fig = go.Figure()
        empty_fig.update_layout(
            **DARK_LAYOUT,
            annotations=[dict(text='No data for selected filters',
                              x=0.5, y=0.5, showarrow=False,
                              font=dict(color=TEXT_COLOR, size=16))],
        )
        kpi = make_kpi_cards(df)  # show totals on empty
        return kpi, empty_fig, empty_fig, empty_fig, empty_fig, empty_fig, empty_fig

    return (
        make_kpi_cards(filtered),
        make_map(filtered),
        make_timeline(filtered, freq),
        make_comparison(filtered),
        make_event_dist(filtered),
        make_fatality(filtered),
        make_source_heatmap(filtered),
    )

print(" Callbacks registered")

 Callbacks registered


In [1]:
# Kill any existing process on port 8050
import subprocess
subprocess.run(['fuser', '-k', '8050/tcp'], capture_output=True)
import time
time.sleep(2)
print(" Port 8050 cleared")

 Port 8050 cleared


In [10]:
# BLOCK 9: ngrok via snap install (most reliable on Colab)

import subprocess, threading, time, json

PORT = 8050
NGROK_TOKEN = "3BlBGNf4FyS8zSSqbEcnyNvMAYe_7Xt4FGusrJxE6RJtz2bir"  # paste your NEW token (reset the old one!)

# Step 1: Install ngrok via apt (official Debian package)
print(" Installing ngrok...")
subprocess.run(['curl', '-sSL', 'https://ngrok-agent.s3.amazonaws.com/ngrok.asc',
                '-o', '/tmp/ngrok.asc'], check=True)
subprocess.run(['bash', '-c',
    'cat /tmp/ngrok.asc | sudo tee /etc/apt/trusted.gpg.d/ngrok.asc > /dev/null'], check=True)
subprocess.run(['bash', '-c',
    'echo "deb https://ngrok-agent.s3.amazonaws.com buster main" | sudo tee /etc/apt/sources.list.d/ngrok.list'], check=True)
subprocess.run(['sudo', 'apt-get', 'update', '-qq'], check=True)
subprocess.run(['sudo', 'apt-get', 'install', '-y', '-qq', 'ngrok'], check=True)
print(" ngrok installed")

# Step 2: Authenticate
subprocess.run(['ngrok', 'config', 'add-authtoken', NGROK_TOKEN], check=True)
print(" ngrok authenticated")

# Step 3: Start Dash in background
def run_app():
    app.run(port=PORT, debug=False, use_reloader=False)

thread = threading.Thread(target=run_app, daemon=True)
thread.start()
time.sleep(4)
print(" Dash server running")

# Step 4: Start ngrok tunnel
subprocess.Popen(['ngrok', 'http', str(PORT)],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)

# Step 5: Get public URL from ngrok API
result = subprocess.run(['curl', '-s', 'http://localhost:4040/api/tunnels'],
                        capture_output=True, text=True)
tunnels = json.loads(result.stdout)
public_url = tunnels['tunnels'][0]['public_url']

print("=" * 60)
print(f"   Dashboard URL: {public_url}")
print("   Click above to open your dashboard!")
print("=" * 60)

 Installing ngrok...
 ngrok installed
 ngrok authenticated
Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:8050
INFO:werkzeug:Press CTRL+C to quit


 Dash server running
   Dashboard URL: https://cleta-acroterial-ornerily.ngrok-free.dev
   Click above to open your dashboard!
